# 🏎️ Notebook 11 — The Clone Army
### Distributed RL, Multiprocessing, & A3C

**Series**: RL Notebook Series · Act IV — Engineering · Post 11 of 16

---

## The Wall of Time

If you just finished fighting for your life to perfectly tune Notebook 10's Continuous PPO, you learned an important lesson about Deep Reinforcement Learning:
**It requires a massive amount of data to learn physics.**

In physics tasks, 5000 episodes could take anywhere from 10 minutes to an hour of pure *Wall-Clock Time* on a single CPU core. And that's just a simple 2D pendulum! Imagine trying to train a 3D robot arm to solve a Rubik's cube, or an agent playing Dota 2.

If an environment takes `50ms` to step the physics engine, a single agent is physically bottlenecked. It can never gather more than 20 transitions per second. Your massively powerful GPU sits completely idle `90%` of the time waiting for that single CPU core to slowly tick the physics simulation forward.

## The Solution: The Clone Army (Distributed RL)

If we have a processor with 16 cores, why not launch **16 environments simultaneously**? We can clone the Agent 16 times, hand each clone their own environment, and tell them to go play independently. 

Instead of collecting 1 trajectory per hour, we collect 16. We then merge all 16 trajectories together into one massive batch and hand it to the GPU to update the "Global Brain".

This concept is the backbone of almost all modern State-of-the-Art RL algorithms.

### 1. Asynchronous Updates (A3C)
In DeepMind's original 2016 A3C paper (Asynchronous Advantage Actor-Critic), they introduced the concept of a true **Global Network**:
1. You create **1 Global Network** that sits on your main CPU thread.
2. You spawn **$N$ Worker Processes** on $N$ different CPU cores. Each Worker has its own environment AND its own personal copy of the neural network (Local Networks).
3. A Worker plays the game for `128` steps and calculates its PPO Loss.
4. Instead of updating its own weights, the Worker "pushes" its computed gradients asynchronously to the Global Network (like pushing code to GitHub).
5. The Global Network uses those gradients to update its Master Weights.
6. The Worker then "pulls" the newest Master Weights, overwriting its Local Network, and repeats.

*Because it is Asynchronous, no worker ever waits for another worker to finish. Worker #4 might push its gradients 2 milliseconds later than Worker #7. They just constantly blast gradients at the Master brain independently!*

### 2. Synchronous Updates (A2C & Vectorized Environments)
The code we will write in this notebook uses **Synchronous Updates** (A2C). 
Instead of independent workers pushing chaotic gradients at random intervals, we literally stack the 8 environments side-by-side into a neat matrix.
1. When `env.reset()` fires, it gives us a perfectly stacked array of 8 different starting states.
2. We pass all 8 states through our **single** Neural Network at exactly the same time.
3. The network spits out 8 different actions synchronously.
4. We wait for all 8 environments to finish stepping, collect the rewards, and step again.

Vectorized environments are infinitely easier to write, less prone to crashing your computer, and provide massive, perfectly formatted batches of training data for the GPU to update on instantly!

### 3. IMPALA (Importance Weighted Actor-Learner Architecture)
A3C was revolutionary, but shipping massive weight matrices around memory between cores was slow. 
DeepMind later introduced **IMPALA**. They completely decoupled the "Actors" from the "Learner":
- 100s of **Actors** run entirely on cheap CPUs. They don't calculate gradients. They just take the current model, run the game blazingly fast, and stream the resulting `(State, Action, Reward)` observations through a highly compressed network pipe directly into a central Queue.
- A single massive **Learner** (the GPU) just reads the Queue non-stop, mathematically correcting for the fact that the Actors are using slightly "older" versions of the model (called *V-Trace* off-policy correction), and updates the weights 24/7.

This architecture trained AlphaStar (StarCraft II) and OpenAI Five (Dota 2)!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import time

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3})

torch.manual_seed(42)
np.random.seed(42)

## 2. Gymnasium's Vectorized Environments

Implementing raw multiprocessing `pipes` and queues in Python is outside the scope of RL theory (and usually requires a heavy framework like `Ray.io` or `RLlib`).

Thankfully, Gymnasium provides a brilliant wrapper called `gym.make_vec`. 
It automatically spawns $N$ background processes, hands each one a `CartPole-v1` environment, and wraps them all up to look like a single synchronized environment.

When we call `env.reset()`, it returns an array of shape `(8, 4)` (8 pending games, 4 features each)!
When we call `env.step([act1, act2...])`, we pass it a list of 8 actions, and it steps all 8 games instantaneously in parallel.

In [ ]:
class PPOActorCritic(nn.Module):
    # Exactly the identical class from Notebook 09
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(PPOActorCritic, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.actor_fc = nn.Linear(hidden_dim, action_dim)
        self.critic_fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, state):
        x = torch.tanh(self.fc1(state))
        action_logits = self.actor_fc(x)
        action_probs = F.softmax(action_logits, dim=-1)
        state_value = self.critic_fc(x)
        return action_probs, state_value
    
    def evaluate(self, state, action):
        action_probs, state_value = self.forward(state)
        m = Categorical(action_probs)
        log_prob = m.log_prob(action)
        entropy = m.entropy()
        return log_prob, state_value, entropy

## 3. The Vectorized PPO Loop

Look incredibly closely at this loop. We no longer wait until an episode finishes (`done == True`).
Instead, we just step all 8 clones exactly `steps_per_env` times (e.g. 128).
If any of the 8 clones crash their cart halfway through, the vectorized environment **automatically resets** that specific clone and sends back the new `state`, completely completely transparently under the hood!

This means we ALWAYS gather precisely $128 \times 8 = 1024$ perfectly formatted transitions every single loop, regardless of how good or bad the agents are performing.

In [ ]:
def train_sync_vector_ppo(env_name="CartPole-v1", num_envs=8, total_timesteps=150_000, steps_per_env=128):
    # Launch 8 simultaneous worker environments using Python Multiprocessing via Gymnasium!
    # TODO: Create the Vectorized Environment!
    # Use gym.make_vec to launch `num_envs` parallel instances of `env_name`.
    envs = None
    
    # Note: Vector env spaces are wrapped, so we explicitly pull the shape of the sub-environment
    state_dim = envs.single_observation_space.shape[0]
    action_dim = envs.single_action_space.n
    
    network = PPOActorCritic(state_dim, action_dim)
    optimizer = optim.Adam(network.parameters(), lr=2.5e-3, eps=1e-5)
    
    # We pre-allocate massive tensors to hold everything. This avoids slow Python lists.
    # Shape: (128 steps, 8 parallel clones, ...)
    obs = torch.zeros((steps_per_env, num_envs, state_dim))
    actions = torch.zeros((steps_per_env, num_envs))
    logprobs = torch.zeros((steps_per_env, num_envs))
    rewards = torch.zeros((steps_per_env, num_envs))
    dones = torch.zeros((steps_per_env, num_envs))
    values = torch.zeros((steps_per_env, num_envs))
    
    global_step = 0
    start_time = time.time()
    returns_tracker = []
    time_tracker = []
    ep_returns = np.zeros(num_envs)
    
    # Initial reset returns 8 starting states: Shape (8, 4)
    next_obs, _ = envs.reset(seed=42)
    next_obs = torch.Tensor(next_obs)
    next_done = torch.zeros(num_envs)
    
    num_updates = total_timesteps // (steps_per_env * num_envs)
    
    for update in range(1, num_updates + 1):
        # 1. Rollout Phase (The 8 Clones Play)
        for step in range(0, steps_per_env):
            global_step += num_envs
            obs[step] = next_obs
            dones[step] = next_done
            
            # Batch inference! The Actor calculates 8 actions instantaneously.
            with torch.no_grad():
                probs, value = network(next_obs)
                dist = Categorical(probs)
                
                # TODO: Sample the action!
                # In Notebook 09, `action` was a single integer. 
                # But because `probs` is shape (8, 2), `.sample()` will natively return an array of 8 actions!
                action = None 
                
                # TODO: Get the log probabilties of the 8 actions.
                logprob = None
                
            values[step] = value.squeeze()
            actions[step] = action
            logprobs[step] = logprob
            
            # TODO: Step all 8 environments physically in parallel!
            # 1. Provide the environment the `action` array (remember to convert it to a numpy array!).
            # 2. Extract next_obs_np, reward, terminated, truncated, infos.
            next_obs_np, reward, terminated, truncated, infos = None, None, None, None, None
            
            raise NotImplementedError("Complete the Vectorized PPO loop!")
            
            # Handle raw data
            next_obs = torch.Tensor(next_obs_np)
            rewards[step] = torch.tensor(reward)
            next_done = torch.Tensor(terminated | truncated)
            
            ep_returns += reward
            for i in range(num_envs):
                if terminated[i] or truncated[i]:
                    returns_tracker.append(ep_returns[i])
                    time_tracker.append(time.time() - start_time)
                    ep_returns[i] = 0.0
                        
        # 2. Advantage Estimation (using Generalized Advantage Estimation GAE)
        with torch.no_grad():
            _, next_value = network(next_obs)
            next_value = next_value.squeeze()
            
            advantages = torch.zeros_like(rewards)
            lastgaelam = 0
            for t in reversed(range(steps_per_env)):
                if t == steps_per_env - 1:
                    nextnonterminal = 1.0 - next_done
                    nextvalues = next_value
                else:
                    nextnonterminal = 1.0 - dones[t + 1]
                    nextvalues = values[t + 1]
                delta = rewards[t] + 0.99 * nextvalues * nextnonterminal - values[t]
                advantages[t] = lastgaelam = delta + 0.99 * 0.95 * nextnonterminal * lastgaelam
            returns = advantages + values
            
        # 3. Flatten the batch: (128, 8, ...) becomes (1024, ...)
        b_obs = obs.reshape((-1, state_dim))
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape(-1)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        
        # Normalize the advantages identically across all 8 clones
        b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
        
        # 4. Global Network Update (The standard PPO math!)
        for epoch in range(4):
            new_logprobs, new_value, entropy = network.evaluate(b_obs, b_actions)
            new_value = new_value.squeeze()
            
            logratio = new_logprobs - b_logprobs
            ratio = logratio.exp()
            
            pg_loss1 = b_advantages * ratio
            pg_loss2 = b_advantages * torch.clamp(ratio, 1 - 0.2, 1 + 0.2)
            pg_loss = torch.max(-pg_loss1, -pg_loss2).mean() # Negative because optimizer MINIMIZES loss
            
            v_loss = 0.5 * F.mse_loss(new_value, b_returns)
            ent_loss = entropy.mean()
            
            loss = pg_loss - 0.01 * ent_loss + v_loss
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 0.5)
            optimizer.step()
            
        if update % 5 == 0 and len(returns_tracker) > 0:
            avg_ret = np.mean(returns_tracker[-50:])
            fps = int(global_step / (time.time() - start_time))
            print(f"Update {update:3d} | Total Steps: {global_step:6d} | FPS: {fps:4d} | Avg Reward: {avg_ret:.1f}")
            
            if avg_ret >= 495.0:
                solve_time = time.time() - start_time
                print(f"\n\U0001f3af Solved in {solve_time:.1f} seconds at update {update}!")
                envs.close()
                return time_tracker, returns_tracker, solve_time

    envs.close()
    return time_tracker, returns_tracker, time.time() - start_time

## 4. Launching the Clone Army

Notice the **FPS (Frames Per Second)** printed on the right. 
Because 8 processes are stepping physics simultaneously, we collect data almost an order of magnitude faster. Time is no longer an insurmountable wall.

In [ ]:
import time

print("Training 1 Clone...")
times_1, returns_1, solve_time_1 = train_sync_vector_ppo(num_envs=1, total_timesteps=400_000, steps_per_env=128)
      
print("\nTraining 8 Clones...")
times_8, returns_8, solve_time_8 = train_sync_vector_ppo(num_envs=8, total_timesteps=400_000, steps_per_env=128)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1 Clone
ax1.plot(times_1, returns_1, color='#94a3b8', alpha=0.3)
window = 50
if len(returns_1) >= window:
    smoothed_1 = np.convolve(returns_1, np.ones(window)/window, mode='valid')
    ax1.plot(times_1[window-1:], smoothed_1, color='#475569', linewidth=2)
ax1.set_title(f"1 Worker (Solved in {solve_time_1:.1f}s)")
ax1.set_xlabel("Wall-Clock Time (seconds)")
ax1.set_xlim(0, max(solve_time_1, solve_time_8))
ax1.set_ylabel("Total Reward")
ax1.axhline(500, color='gray', linestyle='--', label="Solved")
ax1.set_ylim(0, 550)
ax1.legend()

# Plot 8 Clones
ax2.plot(times_8, returns_8, color='#f43f5e', alpha=0.3)
if len(returns_8) >= window:
    smoothed_8 = np.convolve(returns_8, np.ones(window)/window, mode='valid')
    ax2.plot(times_8[window-1:], smoothed_8, color='#e11d48', linewidth=2)

speedup = solve_time_1 / solve_time_8
ax2.set_title(f"8 Workers (Solved in {solve_time_8:.1f}s) - {speedup:.1f}x Faster!")
ax2.set_xlabel("Wall-Clock Time (seconds)")
ax2.set_xlim(0, max(solve_time_1, solve_time_8))
ax2.axhline(500, color='gray', linestyle='--', label="Solved")
ax2.set_ylim(0, 550)
ax2.legend()

plt.tight_layout()
plt.show()

## Conclusion

Vectorization and Distributed RL are engineering tricks, but they are absolutely essential for making Reinforcement Learning viable in reality.
By breaking out of the single-threaded loop, we solved CartPole to maximum efficiency using 150,000 steps in mere seconds.

**But what happens if we don't have an environment to interact with at all?**
What if we just have a static `.csv` file full of terrible logs recorded from a human driving a real car, and we can't physically spawn 8 clones to drive real cars without crashing and dying?

That is the domain of **Offline RL** (Notebook 12)!